In [6]:
import pandas as pd
from nltk.corpus import stopwords
import gensim
import numpy as np


dataset=pd.read_csv("sms_spam.csv")

print(dataset.head())

   type                                               text
0   ham  Hope you are having a good week. Just checking in
1   ham                            K..give back my thanks.
2   ham        Am also doing in cbe only. But have to pay.
3  spam  complimentary 4 STAR Ibiza Holiday or £10,000 ...
4  spam  okmail: Dear Dave this is your final notice to...


In [7]:
print ("Shape:", dataset.shape, '\n')

Shape: (5559, 2) 



In [8]:
def transformText(text):
    stops = set(stopwords.words("english"))
    # Convert text to lowercase
    text = text.lower()
    # Strip multiple whitespaces
    text = gensim.corpora.textcorpus.strip_multiple_whitespaces(text)
    # Removing all the stopwords
    filtered_words = [word for word in text.split() if word not in stops]
    # Preprocessed text after stop words removal
    text = " ".join(filtered_words)
    # Remove the punctuation
    text = gensim.parsing.preprocessing.strip_punctuation(text)
    # Strip all the numerics
    text = gensim.parsing.preprocessing.strip_numeric(text)
    # Removing all the words with less than 3 characters
    text = gensim.parsing.preprocessing.strip_short(text, minsize=3)
    # Strip multiple whitespaces
    text = gensim.corpora.textcorpus.strip_multiple_whitespaces(text)
    # Stemming
    return gensim.parsing.preprocessing.stem_text(text)


In [9]:
myMessage="complimentary 4 STAR Ibiza Holiday or £10,000 cash needs your URGENT collection. 09066364349 NOW from Landline not to lose out! Box434SK38WP150PPM18+"


In [10]:
m1=transformText(myMessage)
print(m1)

complimentari star ibiza holidai cash need urgent collect landlin lose out boxskwpppm


In [11]:
dataset['text'] = dataset['text'].map(transformText)


In [12]:
print(dataset.head())

   type                                               text
0   ham                               hope good week check
1   ham                                    give back thank
2   ham                                  also cbe onli pai
3  spam  complimentari star ibiza holidai cash need urg...
4  spam  okmail dear dave final notic collect tenerif h...


In [13]:
## Split the data
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(dataset['text'], dataset['type'],
                                                    test_size=0.33, random_state=10)

print ("Training Sample Size:", len(X_train), ' ', "Test Sample Size:" ,len(X_test))

Training Sample Size: 3724   Test Sample Size: 1835


In [14]:
#Build the counting corpus
from sklearn.feature_extraction.text import CountVectorizer
count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(X_train)

## Get the TF-IDF vector representation of the data
from sklearn.feature_extraction.text import TfidfTransformer
tfidf_transformer = TfidfTransformer()
X_train_tfidf = tfidf_transformer.fit_transform(X_train_counts)
print ('Dimension of TF-IDF vector :' , X_train_tfidf.shape)

Dimension of TF-IDF vector : (3724, 5056)


In [15]:
#Creating the classifier
#MultinomialNB accepts weights instead of Boolean
from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB()
# the fit() function of any classifier takes the features from the 
# training set X_train_tfidf and the labels from the training set
# y_train
clf.fit(X_train_tfidf, y_train)

MultinomialNB()

In [16]:
print(X_test)

2469                           neva tell noe home aft wat
4107    she good wonder wont sai she smile now cope lo...
1722    sometim put wall around heart not safe get hur...
3428    secret admir reveal think special call opt rep...
4747    thing interest good birthdai wrking nxt start ...
                              ...                        
1778                                        lol come idea
2330                  go orchard lareadi reach soon reach
2803    freedai sexi georg dai pic jordan txt pic dont...
3335                        weekend fine excus much decor
4704    alfi moon children need song mob tell txt tone...
Name: text, Length: 1835, dtype: object


In [17]:
#indexing the test set
X_new_counts = count_vect.transform(X_test)
X_new_tfidf = tfidf_transformer.transform(X_new_counts)

In [18]:
print(X_new_tfidf)

  (0, 4775)	0.345211094285538
  (0, 4314)	0.32727101820059346
  (0, 2938)	0.4754621749532086
  (0, 2903)	0.48160214786703703
  (0, 1990)	0.3230777942250524
  (0, 72)	0.45971601153910663
  (1, 4922)	0.2648009870417136
  (1, 4921)	0.25928050119041257
  (1, 3965)	0.2486198454369446
  (1, 3840)	0.597701057301436
  (1, 3697)	0.206431742876021
  (1, 2970)	0.17549382474899985
  (1, 2500)	0.2559569376038843
  (1, 1765)	0.1824830179587063
  (1, 1167)	0.36835412633571624
  (1, 918)	0.36835412633571624
  (2, 4754)	0.5682520180029045
  (2, 4005)	0.24239610199569778
  (2, 3694)	0.2379481316000987
  (2, 3449)	0.20580099997367562
  (2, 2957)	0.20913480973931176
  (2, 2052)	0.20580099997367562
  (2, 1921)	0.19868678318374292
  (2, 1772)	0.28412600900145224
  (2, 1716)	0.23624118098570215
  :	:
  (1832, 2267)	0.2617897076746588
  (1832, 1713)	0.27485517424757505
  (1832, 1622)	0.27485517424757505
  (1832, 1400)	0.17113203346140243
  (1832, 1206)	0.15011189450878173
  (1832, 1019)	0.1298559828091299
  (

In [19]:
predicted = clf.predict(X_new_tfidf)

In [20]:
print(predicted)

['ham' 'ham' 'ham' ... 'ham' 'ham' 'spam']


In [21]:
predicted==y_test

2469     True
4107     True
1722     True
3428     True
4747     True
        ...  
1778     True
2330     True
2803    False
3335     True
4704     True
Name: type, Length: 1835, dtype: bool

In [22]:
print(np.mean(predicted==y_test))

0.9607629427792915
